In [1]:
import re
import pinyin_jyutping_sentence

In [14]:

def convert_char_to_jyutping(text_list, polyphone=True):
    final_text_list = []
    for text in text_list:
        
        flag = True if re.findall('([\u4e00-\u9fa5]) +([\u4e00-\u9fa5])', text) else False
        while flag:
            text = re.sub("([\u4e00-\u9fa5]) +([\u4e00-\u9fa5])", lambda x:x.group(1)+x.group(2), text)
            flag = True if re.findall('([\u4e00-\u9fa5]) +([\u4e00-\u9fa5])', text) else False
        text = re.sub('([\u4e00-\u9fa5])([a-zA-Z])', lambda x:x.group(1)+" "+x.group(2), text)
        text = re.sub('([a-zA-Z])([\u4e00-\u9fa5])', lambda x:x.group(1)+" "+x.group(2), text)
        jyutpings = pinyin_jyutping_sentence.jyutping(text, spaces=True,tone_numbers=True)
        char_list = [' ']
        for item in jyutpings.split('  '):
            if not re.sub('[a-zA-Z ]', '', item):
                char_list.extend(item.split())
                char_list.append(' ')
            else:
                for subitem in item.split():
                    char_list.append(subitem)
                    char_list.append(' ')

        final_text_list.append(char_list)

    return final_text_list

text = "二零七四一八二八 Amani Great 二零七四一八二八"
result = convert_char_to_jyutping([text])
print(result)

ji6 ling4 cat1 sei3 jat1 baat3 ji6 baat3   A m a n i   G r e a t   ji6 ling4 cat1 sei3 jat1 baat3 ji6 baat3
['ji6 ling4 cat1 sei3 jat1 baat3 ji6 baat3', ' A m a n i', ' G r e a t', ' ji6 ling4 cat1 sei3 jat1 baat3 ji6 baat3']
[[' ', 'ji6', ' ', 'ling4', ' ', 'cat1', ' ', 'sei3', ' ', 'jat1', ' ', 'baat3', ' ', 'ji6', ' ', 'baat3', ' ', 'A', 'm', 'a', 'n', 'i', ' ', 'G', 'r', 'e', 'a', 't', ' ', 'ji6', ' ', 'ling4', ' ', 'cat1', ' ', 'sei3', ' ', 'jat1', ' ', 'baat3', ' ', 'ji6', ' ', 'baat3', ' ']]


In [13]:
import jieba

from pypinyin import lazy_pinyin, Style

def convert_char_to_pinyin(text_list, polyphone=True):
    if jieba.dt.initialized is False:
        jieba.default_logger.setLevel(50)  # CRITICAL
        jieba.initialize()

    final_text_list = []
    custom_trans = str.maketrans(
        {";": ",", "“": '"', "”": '"', "‘": "'", "’": "'"}
    )  # add custom trans here, to address oov

    def is_chinese(c):
        return (
            "\u3100" <= c <= "\u9fff"  # common chinese characters
        )

    for text in text_list:
        char_list = []
        text = text.translate(custom_trans)
        for seg in jieba.cut(text):
            seg_byte_len = len(bytes(seg, "UTF-8"))
            if seg_byte_len == len(seg):  # if pure alphabets and symbols
                if char_list and seg_byte_len > 1 and char_list[-1] not in " :'\"":
                    char_list.append(" ")
                char_list.extend(seg)
            elif polyphone and seg_byte_len == 3 * len(seg):  # if pure east asian characters
                seg_ = lazy_pinyin(seg, style=Style.TONE3, tone_sandhi=True)
                for i, c in enumerate(seg):
                    if is_chinese(c):
                        char_list.append(" ")
                    char_list.append(seg_[i])
            else:  # if mixed characters, alphabets and symbols
                for c in seg:
                    if ord(c) < 256:
                        char_list.extend(c)
                    elif is_chinese(c):
                        char_list.append(" ")
                        char_list.extend(lazy_pinyin(c, style=Style.TONE3, tone_sandhi=True))
                    else:
                        char_list.append(c)
        final_text_list.append(char_list)

    return final_text_list
text = "二零七四一八二八 Amani Great 二零七四一八二八"
result = convert_char_to_pinyin([text])
print(result)

[[' ', 'er4', ' ', 'ling2', ' ', 'qi1', ' ', 'si4', ' ', 'yi1', ' ', 'ba1', ' ', 'er4', ' ', 'ba1', ' ', 'A', 'm', 'a', 'n', 'i', ' ', 'G', 'r', 'e', 'a', 't', ' ', ' ', 'er4', ' ', 'ling2', ' ', 'qi1', ' ', 'si4', ' ', 'yi1', ' ', 'ba1', ' ', 'er4', ' ', 'ba1']]
